In [ ]:
#This is pyenv file for making AI classification.

In [1]:
import os
import pandas as pd
from IPython.display import display, Markdown
from tqdm import tqdm # makes pretty progress bars
tqdm.pandas()  
from dotenv import load_dotenv
load_dotenv()
import time

In [2]:
data=pd.read_csv('data2020.csv')

In [3]:
data_dir=''
SIZE_OF_SAMPLE_FOR_HANDCODING = 50
text_column_name = "DESC_DEFECT"
data.dropna(subset=[text_column_name], inplace=True)

In [4]:
data_sample = data.sample(n=SIZE_OF_SAMPLE_FOR_HANDCODING, random_state=613)

data_sample = data_sample.copy() # avoid annoying warnings!
data_sample.head()

,RECORD_ID,CAMPNO,MAKETXT,MODELTXT,YEARTXT,MFGCAMPNO,COMPNAME,MFGNAME,BGMAN,ENDMAN,...,DESC_DEFECT,CONEQUENCE_DEFECT,CORRECTIVE_ACTION,NOTES,RCL_CMPT_ID,MFR_COMP_NAME,MFR_COMP_DESC,MFR_COMP_PTNO,PARK_IT,PARK_OUTSIDE
16314,223277,22V365000,MERCEDES-BENZ,GLC 350,2020,See Recall Schedule,ELECTRICAL SYSTEM:SOFTWARE,"Mercedes-Benz USA, LLC",NaN,NaN,...,"Mercedes-Benz USA, LLC (MBUSA) is recalling ce...",A disabled eCall system would prevent a vehicl...,The communication module software will be upda...,Owners may also contact the National Highway T...,120389004706752000001105,SW communication module,SW communication module,A2239026409,No,No
4543,160979,21V005000,SMEAL,AERIAL,2020,20-17,EQUIPMENT,"Smeal Holding, LLC",NaN,NaN,...,"Smeal Holding, LLC (Smeal) is recalling certai...",Inability to use the wireless remote controlle...,"Smeal will notify owners, and dealers will ask...",Owners may also contact the National Highway T...,107988004676749000000329,NaN,NaN,NaN,No,No
11204,189397,21V058000,MERCEDES-BENZ,AMG G63,2020,2021020025,COMMUNICATION: AUTO CRASH NOTIFICATION,"Mercedes-Benz USA, LLC",NaN,NaN,...,"Mercedes-Benz USA, LLC (MBUSA) is recalling ce...",Failure of the eCall system may result in emer...,"MBUSA will notify owners, and the communicatio...",Owners may also contact the National Highway T...,108471004318718000001420,SW HERMES,Control Module Unit Software,A2229023217,No,No
17394,228076,22V365000,MERCEDES-BENZ,GLB 250,2020,See Recall Schedule,COMMUNICATION: AUTO CRASH NOTIFICATION,"Mercedes-Benz USA, LLC",NaN,NaN,...,"Mercedes-Benz USA, LLC (MBUSA) is recalling ce...",A disabled eCall system would prevent a vehicl...,The communication module software will be upda...,Owners may also contact the National Highway T...,120389004308717000001420,SW communication module,SW communication module,A2229027218,No,No
14286,211117,22V365000,MERCEDES-BENZ,GLS 450,2020,See Recall Schedule,ELECTRICAL SYSTEM:SOFTWARE,"Mercedes-Benz USA, LLC",NaN,NaN,...,"Mercedes-Benz USA, LLC (MBUSA) is recalling ce...",A disabled eCall system would prevent a vehicl...,The communication module software will be upda...,Owners may also contact the National Highway T...,120389004181720000001105,SW communication module,SW communication module,A2479025009,No,No


In [5]:
from openrouter import OpenRouter
openrouter_client = OpenRouter(api_key=os.environ.get("OPENROUTER_API_KEY_LEDE3"))

In [6]:
prompt_base = """

This is a data car recall data Summary from NHTSA.
Please read this and classify what caused the recall
, choosing from just these options: {categories}. Return only the topic name,
nothing else.
if it is caused by software issue, please select 'software' first.
Software includes driving assist system(ADAS).

Summary is {Summary}

"""

from enum import Enum, IntEnum
class RecallOptions(str, Enum):
    Software = "software"
    Safety_Equipment = "safety_equipment"
    Powertrain = "powertrain"
    Chasies = "chasies"
    Electronics="electronics"
    other_reason="others"
    
data_sample_prompt_column = data_sample.apply(lambda row: prompt_base.format(
    Summary=row[text_column_name],
    categories=", ".join(['"{}"'.format(opt.value) for opt in RecallOptions])
), axis="columns")
full_data_prompt_column = data.apply(lambda row: prompt_base.format(
    Summary=row[text_column_name],
    categories=", ".join(['"{}"'.format(opt.value) for opt in RecallOptions])
), axis="columns")

In [7]:
from strip_tags import strip_tags
import tiktoken
def count_tokens(model, text):
  if "claude" in model and os.environ.get("ANTHROPIC_API_KEY_LEDE"):
    return anthropic_client.messages.count_tokens(model=model, messages=[{"content": text, "role": "user"}])
  elif "gpt" in model:
    encoding = tiktoken.encoding_for_model('gpt-4o') # most modern models use the same tokenizer
    tokens = encoding.encode(text)
    return len(tokens)
  else:
    return count_tokens("gpt-4o", text) # for estimation purposes, pretend mistral is chatgpt...

INPUT_TOKEN_COSTS = {
    "gpt-5-mini": 0.40 / 1_000_000,  # https://openai.com/api/pricing/
    "gpt-5.4-mini": 0.75 / 1_000_000,
    "gpt-5.4": 2.5 / 1_000_000,
    "gpt-5.5": 5 / 1_000_000,
    "gpt-5.4-nano": 0.2 / 1_000_000,
    "mistral-medium-latest": 1.5 / 1_000_000, # https://mistral.ai/pricing#api-pricing
    "mistral-large-latest":  0.5 / 1_000_000,  # No, I don't know why medium is more expensive than large.
    "claude-haiku-4.5": 1 / 1_000_000,
    "claude-sonnet-4.6": 3 / 1_000_000
}
# ignores outputs, since the model will output fairly little text

def estimate_cost(model, token_count):
    return token_count * INPUT_TOKEN_COSTS[model]

In [8]:
DEFAULT_MODEL_TO_USE = 'gpt-5.4-mini'
display(Markdown("Pick a model to use: \n" + "\n - ".join(list(INPUT_TOKEN_COSTS.keys()))))
MODEL_TO_USE = None

while MODEL_TO_USE not in INPUT_TOKEN_COSTS.keys():
  MODEL_TO_USE = input(f"type a model name (press enter for default: {DEFAULT_MODEL_TO_USE}): ").strip()
  if MODEL_TO_USE == "":
    MODEL_TO_USE = DEFAULT_MODEL_TO_USE

Pick a model to use: 
gpt-5-mini
 - gpt-5.4-mini
 - gpt-5.4
 - gpt-5.5
 - gpt-5.4-nano
 - mistral-medium-latest
 - mistral-large-latest
 - claude-haiku-4.5
 - claude-sonnet-4.6

type a model name (press enter for default: gpt-5.4-mini):  


In [9]:
count_tokens_for_our_model = lambda text: count_tokens(MODEL_TO_USE, text)

token_count_sample = count_tokens_for_our_model("SAMPLE RESPONSE SAMPLE RESPONSE".join(data_sample_prompt_column))
"Sample would cost: ${:.2f}".format(estimate_cost(MODEL_TO_USE, token_count_sample))

'Sample would cost: $0.01'

In [10]:
## cost estimation for full dataset
token_count_full = count_tokens_for_our_model("SAMPLE RESPONSE SAMPLE RESPONSE".join(full_data_prompt_column))
"Full dataset would cost: ${:.2f}".format(estimate_cost(MODEL_TO_USE, token_count_full))

'Full dataset would cost: $3.43'

In [11]:
from pydantic import BaseModel


class RecallValidOptions(BaseModel):
  classification: RecallOptions

SYSTEM_PROMPT = "You are a helpful assistant"

def chatgpt_classify(prompt_including_tweet):
    # put our prompt into the blob that OpenAI expects
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": prompt_including_tweet,
        }
    ]
    chat_completion = openai_client.responses.parse(
        input=messages,
        model=MODEL_TO_USE,
        text_format=MuskTweetValidOptions,
    )

    # get the answer out of the blob that OpenAI returns.
    resp = chat_completion.output_parsed.classification.value if chat_completion.output_parsed else None
    return resp

openrouter_model_names = {
    "gpt-5-mini": "openai/gpt-5-mini",
    "gpt-5.4-mini": "openai/gpt-5.4-mini",
    "gpt-5.4": "openai/gpt-5.4",
    "gpt-5.5": "openai/gpt-5.5",
    "gpt-5.4-nano": "openai/gpt-5.4-nano",
    "mistral-medium-latest": "mistralai/mistral-medium-3-5",
    "mistral-large-latest": "mistralai/mistral-large-2512",
    "claude-haiku-4.5": "anthropic/claude-4.5-haiku",
    "claude-sonnet-4.6": "anthropic/claude-4.6-sonnet"
}

def openrouter_classify(prompt_including_tweet):
    response = openrouter_client.chat.send(
        model=openrouter_model_names[MODEL_TO_USE],
        messages=[
            {"role": "user", "content": prompt_including_tweet}
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "recall_classification",
                "strict": True,
                "schema": RecallValidOptions.model_json_schema(),
            },
        },
        max_completion_tokens=64,
        temperature=0,
    )
    return  RecallValidOptions.model_validate_json(
        response.choices[0].message.content
    ).classification.value

def classify(prompt_including_tweet):
  "A wrapper for the provider-specific classification functions"
  if USE_OPENROUTER:
    return openrouter_classify(prompt_including_tweet)
  else:
    if "mistral" in MODEL_TO_USE == "mistral-medium-latest":
      return mistral_classify(prompt_including_tweet)
    elif "claude" in MODEL_TO_USE:
      return anthropic_classify(prompt_including_tweet)
    elif "gpt" in MODEL_TO_USE:
      return chatgpt_classify(prompt_including_tweet)
    else:
      raise ValueError("Unknown model")

def classify_fix(prompt_including_tweet):
    for attempt in range(3):
        try:
            result = classify(prompt_including_tweet)

            time.sleep(0.5)

            return result
        except Exception as e:
            time.sleep(attempt*3)
    return None    

In [12]:
USE_OPENROUTER=True
# ask ChatGPT about each and every tweet IN THE SAMPLE, using the prompt we made above
data_sample["ai_guess"] = data_sample_prompt_column.progress_apply(classify_fix)

100%|████████████████████████████████████████████████████████████████████████████████████| 50/50 [01:15<00:00,  1.51s/it]


In [13]:
with pd.option_context("display.max_colwidth", 500):
  display(
      data_sample[[text_column_name, "ai_guess"]].head(10)
  )

,DESC_DEFECT,ai_guess
16314,"Mercedes-Benz USA, LLC (MBUSA) is recalling certain 2017-2022 CLA-Class, GLA-Class, GLE-Class, GLS-Class, SLC-Class, A-Class, AMG GT-Class, C-Class, E-Class, S-Class, SL-Class, GLC-Class, CLS-Class, GLB-Class, and G-Class vehicles. Please refer to MBUSA's recall report for specific vehicle model details. An error in the communication module's SIM card software can cause a mobile network connection failure, disabling the emergency call (eCall) system.",software
4543,"Smeal Holding, LLC (Smeal) is recalling certain 2020 Aerial and Pumper vehicles, equipped with battery-powered 3600 wireless remote controller units used to control the movement of electric firefighting monitors in firefighting vehicles. The monitor is used to direct the water stream and adjust the spray pattern. The wireless remote controller's sleep mode is disabled, which results in the batteries discharging faster than normal.",electronics
11204,"Mercedes-Benz USA, LLC (MBUSA) is recalling certain 2016 - 2021 CLA-Class, GLA-Class, GLE-Class, GLS-Class, SLC-Class, A-Class, GT-Class, C-Class, E-Class, S-Class, CLS-Class, SL-Class, B-Class, GLB-Class, GLC-Class, and G-Class vehicles. Please refer to the Part 573 report for specific vehicle model details. The software design of the communication module may fail to communicate the correct vehicle location for the emergency call system (eCall) in the event of a crash.",software
17394,"Mercedes-Benz USA, LLC (MBUSA) is recalling certain 2017-2022 CLA-Class, GLA-Class, GLE-Class, GLS-Class, SLC-Class, A-Class, AMG GT-Class, C-Class, E-Class, S-Class, SL-Class, GLC-Class, CLS-Class, GLB-Class, and G-Class vehicles. Please refer to MBUSA's recall report for specific vehicle model details. An error in the communication module's SIM card software can cause a mobile network connection failure, disabling the emergency call (eCall) system.",software
14286,"Mercedes-Benz USA, LLC (MBUSA) is recalling certain 2017-2022 CLA-Class, GLA-Class, GLE-Class, GLS-Class, SLC-Class, A-Class, AMG GT-Class, C-Class, E-Class, S-Class, SL-Class, GLC-Class, CLS-Class, GLB-Class, and G-Class vehicles. Please refer to MBUSA's recall report for specific vehicle model details. An error in the communication module's SIM card software can cause a mobile network connection failure, disabling the emergency call (eCall) system.",software
6377,"Honda (American Honda Motor Co.) is recalling certain 2019-2020 Acura MDX, MDX Sport Hybrid, RDX, TLX, Honda Accord, Civic Hatchback, Insight, 2019 Acura ILX, Honda Accord Hybrid, Civic Coupe, Civic Coupe Si, Civic Sedan, Civic Sedan Si, Civic Type R, Fit, HR-V, Odyssey, Passport, Pilot and Ridgeline, and 2018-2019 CR-V vehicles. The low-pressure fuel pump inside the fuel tank may fail.",powertrain
2634,"Chrysler (FCA US LLC) is recalling certain 2020 Jeep Gladiator and Jeep Cherokee, 2019-2020 Ram 1500 Pickup, Ram 2500 Pickup, Ram 3500 Pickup, Chrysler Pacifica, Dodge Durango, Jeep Grand Cherokee, Jeep Wrangler, and Jeep Renegade and 2019 Dodge Challenger vehicles equipped with 8.4"" or 12"" radio displays. A software error can cause the rearview camera image to remain displayed after the vehicle has been shifted out of reverse. As such, these vehicles fail to comply with the requirements o...",software
1939,"Frazer is recalling certain 2019-2020 emergency vehicles built on RAM 4500 and 5500 chassis, equipped with certain Compressible Liquid Adaptive Suspension Systems (C.L.A.S.S.), part numbers DS135RS3A, DS135RS3AF, DS135RS3-DM. The rear brake caliper flexible line may have been routed so that it contacts a suspension component, possibly causing excessive wear and a loss of rear brakes.",chasies
20053,"Honda (American Honda Motor Co.) is recalling certain 2013-2023 Honda Accord, Civic Coupe, Civic Sedan, Civic Hatchback, Civic Type R, CR-V, HR-V, Ridgeline, Odyssey, Acura ILX, MDX, MDX Hybrid, RDX, RLX, TLX, 2019-2022 Honda Insight, Passport, 2020 Honda CR-V Hybrid, 2018-2019 Hond

In [17]:
okay_to_do_the_whole_thing = input()

In [18]:
data_full_prompt_column = prompt_base + data[text_column_name]

In [21]:
data["ai_guess"] = data_full_prompt_column.progress_apply(classify_fix)
data.to_csv(data_dir + 'Recall_classified2020.csv')

  0%|                                                                               | 17/23098 [00:23<8:49:31,  1.38s/it]


KeyboardInterrupt: 

In [45]:
import os
import requests
import json

OPENROUTER_API_KEY = os.environ["OPENROUTER_API_KEY_LEDE1"]

response = requests.get(
    "https://openrouter.ai/api/v1/key",
    headers={
        "Authorization": f"Bearer {OPENROUTER_API_KEY}"
    }
)

print(response.status_code)
print(json.dumps(response.json(), indent=2))

200
{
  "data": {
    "label": "sk-or-v1-58a...9c0",
    "is_management_key": false,
    "is_provisioning_key": false,
    "limit": 20,
    "limit_reset": "daily",
    "limit_remaining": 20,
    "include_byok_in_limit": false,
    "usage": 27.17007345,
    "usage_daily": 0,
    "usage_weekly": 0.04318075,
    "usage_monthly": 23.67982995,
    "byok_usage": 0,
    "byok_usage_daily": 0,
    "byok_usage_weekly": 0,
    "byok_usage_monthly": 0,
    "is_free_tier": false,
    "expires_at": null,
    "creator_user_id": "user_2uatOGNq9lBDlGjBej7lXgXiKT3",
    "rate_limit": {
      "requests": -1,
      "interval": "10s",
      "note": "This field is deprecated and safe to ignore."
    }
  }
}


In [27]:
import os
import requests

key = os.getenv("OPENROUTER_API_KEY_LEDE1")

print(key is None)
print(len(key) if key else None)
print(repr(key[:10]) if key else None)

True
None
None


In [28]:
load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY_LEDE1")

print(OPENROUTER_API_KEY is None)
print(len(OPENROUTER_API_KEY) if OPENROUTER_API_KEY else None)

False
73


In [39]:
results = []

for i, text in enumerate(data_sample_prompt_column.head(50)):
    t0 = time.perf_counter()

    try:
        guess = classify_fix(text)
        err = None
    except Exception as e:
        guess = None
        err = repr(e)

    elapsed = time.perf_counter() - t0

    results.append({
        "i": i,
        "elapsed_sec": elapsed,
        "text_len": len(text),
        "guess": guess,
        "error": err,
    })

    print(i, round(elapsed, 2), "sec", "len=", len(text), "error=", err)

0 9.25 sec len= 850 error= None
1 9.28 sec len= 832 error= None
2 9.24 sec len= 870 error= None
3 9.21 sec len= 850 error= None
4 9.27 sec len= 850 error= None
5 9.23 sec len= 784 error= None
6 9.23 sec len= 969 error= None
7 9.29 sec len= 784 error= None
8 9.27 sec len= 805 error= None
9 9.24 sec len= 850 error= None
10 9.28 sec len= 659 error= None
11 9.25 sec len= 748 error= None
12 9.24 sec len= 612 error= None
13 9.31 sec len= 870 error= None
14 9.29 sec len= 850 error= None
15 9.24 sec len= 1116 error= None
16 9.25 sec len= 841 error= None
17 9.35 sec len= 805 error= None
18 9.37 sec len= 965 error= None
19 9.23 sec len= 865 error= None


KeyboardInterrupt: 

In [40]:
log_df = pd.DataFrame(results)
log_df.sort_values("elapsed_sec", ascending=False).head(10)

,i,elapsed_sec,text_len,guess,error
18,18,9.370734,965,None,None
17,17,9.350907,805,None,None
13,13,9.307574,870,None,None
14,14,9.292512,850,None,None
7,7,9.286719,784,None,None
10,10,9.284997,659,None,None
1,1,9.276575,832,None,None
4,4,9.268969,850,None,None
8,8,9.268227,805,None,None
16,16,9.254144,841,None,None


In [62]:
import time

def classify_fix(prompt_including_tweet):
    last_error = None

    for attempt in range(3):
        try:
            print(f"attempt {attempt + 1}")

            result = classify(prompt_including_tweet)

            time.sleep(0.5)

            return result

        except Exception as e:
            last_error = e
            print(f"attempt {attempt + 1} failed:", repr(e))

            time.sleep(attempt * 3)

    print("all attempts failed:", repr(last_error))
    return None